# 🔮 Notebook 02: Modelagem Preditiva de Séries Temporais (SARIMAX vs ML)
**Disciplina**: Técnicas de IA Aplicadas a Sistemas de Energia  
**Programa**: Mestrado Profissional — IFSC  
**Autor**: Dilson Eijo Rigotti  

---

## 🎯 Objetivo
Desenvolver e comparar modelos preditivos de séries temporais (**SARIMAX** vs **Gradient Boosting ML**) para projetar o consumo de energia elétrica (MWh) do Grupo B em Santa Catarina até a abertura total do Mercado Livre (**2027-2028**), estabelecida pela Lei nº 15.269/2025.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import json

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Carregamento da Série Temporal e Separação Treino/Teste
- **Treino**: 360 meses (1994 a 2023)
- **Teste**: 27 meses (2024 a Março/2026)

In [ ]:
df = pd.read_csv('../data/processed/sc_grupo_b_mensal.csv')
df['data'] = pd.to_datetime(df['data'])

df_res = df[df['classe'] == 'Residencial'].sort_values('data').copy()
df_res.set_index('data', inplace=True)
ts = df_res['consumo_mwh'].asfreq('MS').interpolate(method='linear')

split_date = '2024-01-01'
train = ts[ts.index < split_date]
test = ts[ts.index >= split_date]

print(f"Treino: {len(train)} meses ({train.index[0].strftime('%Y-%m')} a {train.index[-1].strftime('%Y-%m')})")
print(f"Teste:  {len(test)} meses ({test.index[0].strftime('%Y-%m')} a {test.index[-1].strftime('%Y-%m')})")

## 2. Modelo 1: SARIMAX $(1,1,1) \times (1,1,1)_{12}$
Ajuste da classe estatística autorregressiva com sazonalidade estival de 12 meses.

In [ ]:
model_sarimax = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12))
results_sarimax = model_sarimax.fit(disp=False)
pred_sarimax = results_sarimax.predict(start=test.index[0], end=test.index[-1])

## 3. Modelo 2: Machine Learning (Gradient Boosting Regressor)
Criação de variáveis defasadas (Lags 1 a 12) e identificadores de sazonalidade temporal.

In [ ]:
def create_features(series):
    df_feat = pd.DataFrame(index=series.index)
    df_feat['y'] = series.values
    df_feat['mes'] = df_feat.index.month
    df_feat['ano'] = df_feat.index.year
    for lag in range(1, 13):
        df_feat[f'lag_{lag}'] = df_feat['y'].shift(lag)
    return df_feat.dropna()

df_feat_full = create_features(ts)
train_feat = df_feat_full[df_feat_full.index < split_date]
test_feat = df_feat_full[df_feat_full.index >= split_date]

X_train, y_train = train_feat.drop(columns=['y']), train_feat['y']
X_test, y_test = test_feat.drop(columns=['y']), test_feat['y']

model_gb = HistGradientBoostingRegressor(max_iter=200, random_state=42)
model_gb.fit(X_train, y_train)
pred_gb = pd.Series(model_gb.predict(X_test), index=y_test.index)

## 4. Avaliação Comparativa dos Modelos

In [ ]:
with open('../data/processed/model_metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

pd.DataFrame(metrics)

## 5. Projeção de Futuro para o Marco Regulatório (2026-2028)

In [ ]:
future_dates = pd.date_range(start='2026-04-01', periods=33, freq='MS')
model_full = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12))
res_full = model_full.fit(disp=False)
forecast_future = res_full.get_forecast(steps=33)
pred_mean = forecast_future.predicted_mean
conf_int = forecast_future.conf_int()

plt.figure(figsize=(15, 6))
plt.plot(ts['2020-01-01':].index, ts['2020-01-01':].values, label='Consumo Histórico (MWh)', color='black', linewidth=1.8)
plt.plot(test.index, pred_sarimax, label='SARIMAX (Teste 2024-2026)', color='#1f77b4', linestyle='--', linewidth=2)
plt.plot(future_dates, pred_mean, label='Projeção 2026-2028 (Mercado Livre)', color='#e377c2', linewidth=2.2)
plt.fill_between(future_dates, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='#e377c2', alpha=0.2, label='IC 95%')

plt.axvline(pd.to_datetime('2027-11-01'), color='orange', linestyle='--', label='Abertura B3 (Nov/2027)')
plt.axvline(pd.to_datetime('2028-11-01'), color='red', linestyle='--', label='Abertura B1/B2 (Nov/2028)')

plt.title('Projeção Preditiva de Consumo (MWh) - Grupo B SC (2026-2028)', fontsize=14, fontweight='bold')
plt.xlabel('Ano')
plt.ylabel('Consumo Mensal (MWh)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()